# Variedades de manzanas
CNN desde cero. Ejecutá de arriba hacia abajo con GPU. Los pesos quedan en Drive.

Ya hay pesos entrenados en Drive: para probarlos, abrí el notebook 03. Ejecutar este notebook vuelve a entrenar.

In [ ]:
from pathlib import Path
import json
import random
import shutil
import subprocess
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from google.colab import drive

drive.mount('/content/drive')
carpeta = Path('/content/drive/MyDrive/Inspeccion de manzanas - Colab')
carpeta.mkdir(exist_ok=True)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', dispositivo)

In [ ]:
# Descargar solo las cuatro variedades
originales = Path('/content/fruits360_variedades')
variedades = ['apple_crimson_snow_1', 'apple_granny_smith_1',
             'apple_pink_lady_1', 'apple_red_delicios_1']
version = 'e09de71fa3e80624be016e1b5c90c1c310d046c5'
carpetas = []
for parte in ['Training', 'Validation', 'Test']:
    for variedad in variedades:
        carpetas.append(parte + '/' + variedad)

if not originales.exists():
    subprocess.run(['git', 'clone', '--depth=1', '--filter=blob:none', '--sparse', '--no-checkout',
                    'https://github.com/fruits-360/fruits-360-original-size.git',
                    str(originales)], check=True)
subprocess.run(['git', '-C', str(originales), 'sparse-checkout', 'set'] + carpetas, check=True)
subprocess.run(['git', '-C', str(originales), 'fetch', '--depth=1', 'origin', version], check=True)
subprocess.run(['git', '-C', str(originales), 'checkout', '--detach', version], check=True)
print('Fotos listas.')

In [ ]:
# Separar las fotos sin mezclar fotogramas consecutivos
datos = Path('/content/datos_variedades')
if datos.exists():
    shutil.rmtree(datos)
for variedad in variedades:
    fotos = list(originales.glob('*/' + variedad + '/*.jpg'))
    cantidad_r1 = 0
    for foto in fotos:
        if foto.name.startswith('r1_'):
            cantidad_r1 += 1
    mitad = cantidad_r1 // 2

    for foto in fotos:
        rotacion, numero = foto.stem.split('_')
        numero = int(numero)
        if rotacion == 'r0':
            parte = 'entrenamiento'
        elif 16 <= numero < mitad - 16:
            parte = 'validacion'
        elif mitad + 16 <= numero < cantidad_r1 - 16:
            parte = 'prueba'
        else:
            continue
        destino_clase = datos / parte / variedad
        destino_clase.mkdir(parents=True, exist_ok=True)
        shutil.copy2(foto, destino_clase / foto.name)
print('Carpetas de entrenamiento, validación y prueba listas.')

In [ ]:
preparar = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
aumentar = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(45, translate=(0.05, 0.05), scale=(0.85, 1.1), fill=255),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    preparar,
])
entrenamiento = datasets.ImageFolder(datos / 'entrenamiento', transform=aumentar)
validacion = datasets.ImageFolder(datos / 'validacion', transform=preparar)
prueba = datasets.ImageFolder(datos / 'prueba', transform=preparar)
assert entrenamiento.classes == validacion.classes == prueba.classes
lotes_entrenamiento = DataLoader(entrenamiento, batch_size=32, shuffle=True)
lotes_validacion = DataLoader(validacion, batch_size=32)
lotes_prueba = DataLoader(prueba, batch_size=32)
print('Clases:', entrenamiento.classes)
print('Fotos:', len(entrenamiento), len(validacion), len(prueba))

nombres = ['Crimson Snow', 'Granny Smith', 'Pink Lady', 'Red Delicious']
tarea = 'variedades'
ruta_pesos = Path('/content/mejor_variedades.pth')

In [ ]:
modelo = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
    nn.Linear(1024, 64), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(64, len(nombres)),
).to(dispositivo)

error = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001, weight_decay=0.001)
print('Parámetros:', sum(p.numel() for p in modelo.parameters()))

In [ ]:
errores_entrenamiento, errores_validacion = [], []
mejor_error = float('inf')
sin_mejora = 0

for epoca in range(40):
    modelo.train()
    total = 0
    for imagenes, etiquetas in lotes_entrenamiento:
        imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)
        optimizador.zero_grad()
        perdida = error(modelo(imagenes), etiquetas)
        perdida.backward()
        optimizador.step()
        total += perdida.item() * len(imagenes)
    error_train = total / len(entrenamiento)

    modelo.eval()
    total = 0
    with torch.no_grad():
        for imagenes, etiquetas in lotes_validacion:
            imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)
            perdida = error(modelo(imagenes), etiquetas)
            total += perdida.item() * len(imagenes)
    error_val = total / len(validacion)
    errores_entrenamiento.append(error_train)
    errores_validacion.append(error_val)
    print(f'Época {epoca+1:2d} | entrenamiento {error_train:.3f} | validación {error_val:.3f}')

    if error_val < mejor_error - 0.0001:
        mejor_error, mejor_epoca = error_val, epoca + 1
        torch.save(modelo.state_dict(), ruta_pesos)
        sin_mejora = 0
    else:
        sin_mejora += 1
    if sin_mejora == 7:
        break

modelo.load_state_dict(torch.load(ruta_pesos, map_location=dispositivo, weights_only=True))
modelo.eval()
print('Recuperamos la época', mejor_epoca)

In [ ]:
epocas = range(1, len(errores_entrenamiento) + 1)
plt.figure(figsize=(7, 4))
plt.plot(epocas, errores_entrenamiento, label='Entrenamiento con aumentos')
plt.plot(epocas, errores_validacion, label='Validación')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.tight_layout()
plt.savefig(carpeta / (tarea + '_curva.png'))
plt.show()

In [ ]:
resultados = {}
for nombre, lotes in [('validacion', lotes_validacion), ('prueba', lotes_prueba)]:
    reales, predichas = [], []
    with torch.no_grad():
        for imagenes, etiquetas in lotes:
            salida = modelo(imagenes.to(dispositivo))
            reales.extend(etiquetas.tolist())
            predichas.extend(salida.argmax(1).cpu().tolist())
    print(nombre.upper())
    print(classification_report(reales, predichas, labels=range(len(nombres)),
                                target_names=nombres, digits=3, zero_division=0))
    resultados[nombre] = classification_report(reales, predichas, labels=range(len(nombres)),
                                              target_names=nombres, output_dict=True, zero_division=0)

ConfusionMatrixDisplay.from_predictions(reales, predichas, labels=range(len(nombres)),
                                       display_labels=nombres, xticks_rotation=25, colorbar=False)
plt.title('Prueba: etiquetas reales y predicciones')
plt.tight_layout()
plt.savefig(carpeta / (tarea + '_confusion.png'))
plt.show()

In [ ]:
torch.save({'pesos': modelo.cpu().state_dict(), 'clases': nombres,
            'tamano': 128, 'media': [0.5]*3, 'desvio': [0.5]*3},
           carpeta / ('cnn_' + tarea + '.pt'))
resultados['mejor_epoca'] = mejor_epoca
resultados['perdidas'] = {'entrenamiento': errores_entrenamiento, 'validacion': errores_validacion}
with open(carpeta / (tarea + '_metricas.json'), 'w') as archivo:
    json.dump(resultados, archivo, indent=2)
print('Guardado en Drive:', 'cnn_' + tarea + '.pt')

Fruits-360 contiene vistas repetidas de pocas frutas. Separamos las rotaciones y dejamos un margen entre validación y prueba, pero esto no demuestra que funcione con nuevas manzanas en una cinta.

[Datos y licencia](https://github.com/fruits-360/fruits-360-original-size/tree/e09de71fa3e80624be016e1b5c90c1c310d046c5).

Ejecución del 17/11/2026: mejor época 9 de 16. Macro-F1: **0,800 en validación** y **1,000 en prueba**. La diferencia entre ambas particiones y las vistas de las mismas frutas impiden tomar ese 1,000 como rendimiento en escenas nuevas. [Métricas guardadas](https://drive.google.com/file/d/1zvp0f6Oriof1fg8DSIFgX42z83RLHXOT/view).